`LinearFeatures` нельзя один раз `fit_transform`-ить на всём `train_data` перед внешним CV, потому что внутри него есть supervised OOF Target Encoding. Поэтому `LinearFeatures` переносится внутрь `StatsModelsGLM.fit_predict_single_fold`.

Для каждого **outer fold**:

1. `outer valid` полностью исключается из построения признаков;
2. внутри `outer train` строится отдельный `StratifiedGroupKFold`;
3. `LinearFeatures.fit_transform(outer_train)` получает эти inner folds и строит OOF Target Encoding только внутри `outer train`;
4. `LinearFeatures.transform(outer_valid)` использует статистики, fitted только на `outer train`;
5. GLM обучается на OOF-признаках `outer train`;
6. `GROUP (KOD SETI/LPU)` хранится как metadata (`role='group'`) и не попадает в модельную матрицу.

Дополнительно свободный член GLM не штрафуется elastic-net регуляризацией.

In [1]:
import os
import types
import joblib
import warnings

from copy import copy, deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

from lightautoml.tasks import Task
from lightautoml.reader.base import PandasToPandasReader
from lightautoml.pipelines.ml.base import MLPipeline
from lightautoml.pipelines.features.linear_pipeline import LinearFeatures
from lightautoml.ml_algo.base import MLAlgo, TabularMLAlgo
from lightautoml.ml_algo.tuning.optuna import OptunaTuner
from lightautoml.ml_algo.tuning.base import Uniform
from lightautoml.automl.base import AutoML
from lightautoml.report.report_deco import ReportDeco
from lightautoml.dataset.roles import DatetimeRole

try:
    from lightautoml.automl.presets.utils import calc_feats_permutation_imps
except ImportError:
    calc_feats_permutation_imps = None

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

d:\Project\DEV\python_envs\laml_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'nlp' extra dependency package 'fasttext-numpy2' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'nltk' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'transformers' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.


d:\Project\DEV\python_envs\laml_venv\Lib\site-packages\lightautoml\ml_algo\dl_model.py:40: UserWarning: 'transformers' - package isn't installed
  warnings.warn("'transformers' - package isn't installed")
d:\Project\DEV\python_envs\laml_venv\Lib\site-packages\lightautoml\text\embed.py:24: UserWarning: 'transformers' - package isn't installed
  warnings.warn("'transformers' - package isn't installed")
d:\Project\DEV\python_envs\laml_venv\Lib\site-packages\lightautoml\text\dl_transformers.py:25: UserWarning: 'transformers' - package isn't installed
  warnings.warn("'transformers' - package isn't installed")


## 1. Константы


In [2]:
DATA_PATH = r"T:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Виноградов\EmblemConverter\DMS\data_distrib_clean_2.csv"

TARGET_NAME = "PAID_FLAG_1"
GROUP_NAME = "GROUP (KOD SETI/LPU)"
START_DATE_NAME = "START_DATE"

RANDOM_STATE = 109

# Первый group-safe split используется как независимый holdout.
N_SPLITS = 5

# Внешний CV модели.
N_FOLDS = 4

# Внутренний CV только для OOF Target Encoding внутри outer-train.
N_INNER_FOLDS = 4

N_TRIALS = 10
TIMEOUT = 60 * 60 * 6
CHECKPOINT_DIR = "checkpoints"

# DROP_COLUMNS = [
#     # по VIF + overflow
#     "CNT",
#     "CNT_CLINIC_IN_PROGRAM",
#     "ASSISTANCE_TYPE",
#     "OKVED_GROUP",
#     "ATTACHMENT",
#     # мусор
#     "PROGRAM_NAME",
#     "POLICY_NUMBER",
#     # по VIF
#     "INSURANT_NAME",
#     "HOLDING",
#     'PROGRAM_LIST_RISK_PREPAY_SPEC',
#     'PRICE_VALUE',
#     # EVA по 2.2
#     'DOGOVOR_TYPE',
#     'PROLONGATION_TYPE',
#     'UNPINNING',
#     'DIRECT_SERVICE',
#     'PRICE_CLINIC_RANK',
#     'CNT_PROGRAM',
#     'METRO_FLAG',
#     # Удаляем по 0 коэффициентам
#     'CNT_LPU_V_SETI',
#     'MAX_SUBWAY_SYNT',
#     'MIN_BUS_DIST_MIN',
#     'PRICE_LEVEL_NUM',
    
# ]

# Версия 2
DROP_COLUMNS = [
    # overflow
    'CNT',
    'ATTACHMENT',
    # если есть все остальные признаки 0, это странно
    'ASSISTANCE_TYPE',
    # VIF + 0 коэффициенты
    "HOLDING",
    # EVA 2.2
    'METRO_FLAG', # очень большой коэффициент, по сравнению с остальными, PowerStat -, высокий VIF
    'CNT_PROGRAM',
]

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

## 2. Чтение данных

Здесь намеренно **нет `drop_duplicates()` по доступным модели признакам**. Если две строки являются реальными отдельными наблюдениями, одинаковый после удаления `GROUP` вектор признаков не делает одну из строк «лишней» для likelihood.

In [3]:
data = pd.read_csv(DATA_PATH)
print("Исходная форма:", data.shape)

required = [TARGET_NAME, GROUP_NAME]
missing_required = [c for c in required if c not in data.columns]
if missing_required:
    raise KeyError(f"В данных нет обязательных колонок: {missing_required}")

if data[GROUP_NAME].isna().any():
    raise ValueError(
        f"В {GROUP_NAME!r} есть NaN. Перед GroupKFold каждой строке должна быть "
        "однозначно присвоена группа."
    )

data = data.drop(columns=[c for c in DROP_COLUMNS]).copy()

print("После удаления ненужных колонок:", data.shape)
print("Target rate:", data[TARGET_NAME].mean())
print("Уникальных групп:", data[GROUP_NAME].nunique())

Исходная форма: (701975, 43)
После удаления ненужных колонок: (701975, 37)
Target rate: 0.05152747604971687
Уникальных групп: 2750


In [4]:
def count_duplicate_rows(df, feature_cols):
    return df.duplicated(
        subset=feature_cols,
        keep=False
    ).sum()

In [5]:
data['PRICE_LEVEL_NUM'] = data['PRICE_LEVEL_NUM'].fillna(-1)
data['PRICE_VALUE'] = data['PRICE_VALUE'].fillna(-1)
data['COMISSION_PERCENTAGE_FIX']= data['COMISSION_PERCENTAGE_FIX'].fillna(-1)

In [6]:
data['GROUP_SIZE'] = (
    data
    .groupby(GROUP_NAME)[GROUP_NAME]
    .transform('size')
    .astype('int32')
)

In [7]:
print(f"Было строк: {len(data)}")
# Удаляем полные дубликаты (оставляем первое вхождение)
data = data.drop_duplicates(keep='first')

print(f"Стало строк: {len(data)}")


Было строк: 701975
Стало строк: 507045


In [8]:
def duplicate_stats(df: pd.DataFrame) -> dict:
    """Единообразная статистика одинаковых строк."""
    h = pd.util.hash_pandas_object(df, index=False)
    sizes = h.value_counts()

    return {
        "rows": int(len(df)),
        "unique_patterns": int(len(sizes)),
        # Сколько строк являются повторениями сверх первого экземпляра.
        "extra_duplicates": int((sizes - 1).clip(lower=0).sum()),
        # Сколько всего строк входят в группы размера >= 2, включая первые экземпляры.
        "rows_in_duplicate_groups": int(sizes[sizes > 1].sum()),
        "duplicate_groups": int((sizes > 1).sum()),
        "max_duplicate_group_size": int(sizes.max()),
    }


print("Полные строки:")
print(duplicate_stats(data))

raw_model_cols = [
    c for c in data.columns
    if c not in {GROUP_NAME, START_DATE_NAME}
]

print("\nX до LAMA, без group/date:")
print(duplicate_stats(data[raw_model_cols]))

Полные строки:
{'rows': 507045, 'unique_patterns': 507045, 'extra_duplicates': 0, 'rows_in_duplicate_groups': 0, 'duplicate_groups': 0, 'max_duplicate_group_size': 1}

X до LAMA, без group/date:
{'rows': 507045, 'unique_patterns': 505944, 'extra_duplicates': 1101, 'rows_in_duplicate_groups': 2202, 'duplicate_groups': 1101, 'max_duplicate_group_size': 2}


## 3. Независимый holdout по группам

Это соответствует текущей логике: один split `StratifiedGroupKFold` оставляем как полностью внешний test/holdout.

In [9]:
sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    sgkf.split(
        data,
        data[TARGET_NAME],
        groups=data[GROUP_NAME],
    )
)

train_data = data.iloc[train_idx].reset_index(drop=True).copy()
test_data = data.iloc[test_idx].reset_index(drop=True).copy()

print("train_data:", train_data.shape)
print("test_data :", test_data.shape)
print("Target rate train  :", train_data[TARGET_NAME].mean())
print("Target rate holdout:", test_data[TARGET_NAME].mean())

train_groups = set(train_data[GROUP_NAME].unique())
test_groups = set(test_data[GROUP_NAME].unique())
print("Overlap групп train/holdout:", len(train_groups & test_groups))

assert len(train_groups & test_groups) == 0

train_data: (405634, 38)
test_data : (101411, 38)
Target rate train  : 0.06921017468949842
Target rate holdout: 0.06920353807772332
Overlap групп train/holdout: 0


In [10]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
def rank_quadratic_candidates(
    df,
    target_col,
    numeric_cols,
    n_bins=15,
    min_unique=10,
):
    results = []
    for col in numeric_cols:
        tmp = df[[col, target_col]].dropna().copy()
        if tmp[col].nunique() < min_unique:
            continue
        # Квантильные бины по признаку
        q = min(n_bins, tmp[col].nunique())
        try:
            tmp["bin"] = pd.qcut(
                tmp[col],
                q=q,
                duplicates="drop"
            )
        except ValueError:
            continue
        grouped = (
            tmp.groupby("bin", observed=True)
            .agg(
                x_mean=(col, "mean"),
                events=(target_col, "sum"),
                n=(target_col, "size"),
            )
            .reset_index()
        )
        if len(grouped) < 4:
            continue
        # Сглаженная observed probability,
        # чтобы logit не был +/-inf при p=0 или p=1
        grouped["p"] = (
            grouped["events"] + 0.5
        ) / (
            grouped["n"] + 1
        )
        grouped["logit"] = np.log(
            grouped["p"] / (1 - grouped["p"])
        )
        # Нормируем x только для устойчивости
        x = grouped["x_mean"].to_numpy(dtype=float)
        x_mean = np.average(
            x,
            weights=grouped["n"]
        )
        x_std = np.sqrt(
            np.average(
                (x - x_mean) ** 2,
                weights=grouped["n"]
            )
        )
        if x_std == 0:
            continue
        z = (x - x_mean) / x_std
        y = grouped["logit"].to_numpy(dtype=float)
        w = grouped["n"].to_numpy(dtype=float)
        # Линейная зависимость
        X_lin = z.reshape(-1, 1)
        model_lin = LinearRegression()
        model_lin.fit(
            X_lin,
            y,
            sample_weight=w
        )
        pred_lin = model_lin.predict(X_lin)
        # Квадратичная зависимость
        X_quad = np.column_stack([
            z,
            z ** 2
        ])
        model_quad = LinearRegression()
        model_quad.fit(
            X_quad,
            y,
            sample_weight=w
        )
        pred_quad = model_quad.predict(X_quad)
        r2_lin = r2_score(
            y,
            pred_lin,
            sample_weight=w
        )
        r2_quad = r2_score(
            y,
            pred_quad,
            sample_weight=w
        )
        results.append({
            "feature": col,
            "n_unique": tmp[col].nunique(),
            "n_bins": len(grouped),
            "r2_linear": r2_lin,
            "r2_quadratic": r2_quad,
            "delta_r2": r2_quad - r2_lin,
            "quadratic_coef": model_quad.coef_[1],
        })
    return (
        pd.DataFrame(results)
        .sort_values(
            "delta_r2",
            ascending=False
        )
        .reset_index(drop=True)
    )


In [11]:
candidate_cols = train_data.drop(['PAID_FLAG_1', 'INSURED_EXP_NEW', 'SHARE_LPU_IN_PROGRAM'], axis=1).select_dtypes(include='number').columns
candidate_cols

Index(['DAYS', 'UNPINNING', 'CNT_PROGRAM/cnt_policy_beg', 'DIRECT_SERVICE',
       'VIP', 'DOGOVOR_TYPE', 'INS_AGE', 'COVENANTA_ZHUD', 'MAX_REVIEWS',
       'PRICE_LEVEL_NUM', 'PRICE_VALUE', 'PRICE_CLINIC_RANK',
       'COMISSION_PERCENTAGE_FIX', 'SHARE_PRICE_LPU_FROM_MAX',
       'SHARE_REVIEW_FROM_MAX', 'SHARE_BUS_DIST_MIN_FROM_MIN',
       'CNT_CLINIC_IN_PROGRAM', 'diff_RESP_END_START', 'diff_DATE_START_RESP',
       'TIME_TREND', 'MONTH_SIN', 'WEEK_SIN', 'WEEK_COS', 'GROUP_SIZE'],
      dtype='object')

In [12]:
quadratic_rank = rank_quadratic_candidates(
    train_data,
    target_col=TARGET_NAME,
    numeric_cols=candidate_cols,
    n_bins=15,
)
poly_cols = quadratic_rank.loc[(quadratic_rank['delta_r2'] > 0.5), 'feature'].values
poly_cols

array(['SHARE_PRICE_LPU_FROM_MAX'], dtype=object)

In [13]:
def add_quadratic_features(df, cols):
    out = df.copy()

    for col in cols:
        out[f'{col}_sq'] = out[col] ** 2
    return out

In [14]:
train_data = add_quadratic_features(
    train_data,
    poly_cols
)

test_data = add_quadratic_features(
    test_data,
    poly_cols
)

In [ ]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 405633 entries, 0 to 405632
Data columns (total 41 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   GROUP (KOD SETI/LPU)           405633 non-null  object 
 1   INSURANT_NAME                  405633 non-null  object 
 2   START_DATE                     405633 non-null  object 
 3   DAYS                           405633 non-null  int64  
 4   UNPINNING                      405633 non-null  int64  
 5   CNT_PROGRAM/cnt_policy_beg     405633 non-null  float64
 6   Pol                            405633 non-null  object 
 7   DIRECT_SERVICE                 405633 non-null  int64  
 8   VIP                            405633 non-null  int64  
 9   DOGOVOR_TYPE                   394986 non-null  float64
 10  PROLONGATION_TYPE              332987 non-null  object 
 11  INSURANT_REGION                405633 non-null  object 
 12  INSURANT_REGION_GROUP         

## 4. Outer `StratifiedGroupKFold`

Это CV, по которому считаются OOF predictions самой GLM и выбираются гиперпараметры.

In [15]:
cv = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_iter = list(
    cv.split(
        train_data,
        train_data[TARGET_NAME],
        groups=train_data[GROUP_NAME],
    )
)

for fold, (tr_idx, va_idx) in enumerate(cv_iter):
    tr_groups = set(train_data.iloc[tr_idx][GROUP_NAME].unique())
    va_groups = set(train_data.iloc[va_idx][GROUP_NAME].unique())
    overlap = tr_groups & va_groups

    print(
        f"fold={fold}: "
        f"train={len(tr_idx):,}, valid={len(va_idx):,}, "
        f"train_rate={train_data.iloc[tr_idx][TARGET_NAME].mean():.6f}, "
        f"valid_rate={train_data.iloc[va_idx][TARGET_NAME].mean():.6f}, "
        f"group_overlap={len(overlap)}"
    )
    assert not overlap

fold=0: train=304,227, valid=101,407, train_rate=0.069221, valid_rate=0.069177, group_overlap=0
fold=1: train=304,226, valid=101,408, train_rate=0.069228, valid_rate=0.069156, group_overlap=0
fold=2: train=304,228, valid=101,406, train_rate=0.069228, valid_rate=0.069158, group_overlap=0
fold=3: train=304,221, valid=101,413, train_rate=0.069164, valid_rate=0.069350, group_overlap=0


## 5. Task и roles

`GROUP_NAME` больше не имеет role `drop`. Он имеет role `group`: LAMA хранит его как специальный массив `dataset.group`, но не включает в `dataset.features`.

In [16]:
task = Task("binary")

roles = {
    "target": TARGET_NAME,
    "group": GROUP_NAME,
    "drop": [START_DATE_NAME],
}

## 6. Внутренние group-safe folds для Target Encoding

На каждом outer fold inner folds строятся заново **только из outer-train**. Поэтому `y_outer_valid` не может попасть ни в OOF Target Encoding outer-train, ни в статистики encoder-а для outer-valid.

In [17]:
def make_group_safe_fold_ids(
    y,
    groups,
    n_splits=N_INNER_FOLDS,
    random_state=RANDOM_STATE,
):
    y = np.asarray(y).reshape(-1)
    groups = np.asarray(groups)

    unique_groups = pd.Series(groups).nunique(dropna=False)
    n_splits_eff = min(int(n_splits), int(unique_groups))

    if n_splits_eff < 2:
        raise ValueError(
            f"Для inner CV нужно хотя бы 2 уникальные группы; получено {unique_groups}."
        )

    splitter = StratifiedGroupKFold(
        n_splits=n_splits_eff,
        shuffle=True,
        random_state=random_state,
    )

    folds = np.full(len(y), -1, dtype=np.int32)
    dummy_x = np.zeros((len(y), 1), dtype=np.float32)

    for fold, (_, val_idx) in enumerate(
        splitter.split(dummy_x, y, groups=groups)
    ):
        folds[val_idx] = fold

    if (folds < 0).any():
        raise RuntimeError("Не всем строкам присвоен inner fold.")

    # Жёсткая проверка: группа не пересекает inner train/valid.
    for fold in np.unique(folds):
        tr_groups = set(groups[folds != fold])
        va_groups = set(groups[folds == fold])
        if tr_groups & va_groups:
            raise RuntimeError(f"Group leakage во внутреннем fold={fold}.")

    return folds


def attach_folds(dataset, folds):
    """
    Добавляет folds в shallow-copy LAMA Dataset, не мутируя исходный dataset.
    TargetEncoder читает именно dataset.folds.
    """
    ds = copy(dataset)  
    ds.folds = pd.Series(np.asarray(folds, dtype=np.int32), index=ds.data.index)  
  
    attrs = list(getattr(ds, "_array_like_attrs", []))  
    if "folds" not in attrs:  
        attrs.append("folds")  
    ds._array_like_attrs = attrs  
  
    return ds


def make_linear_features():
    """
    auto_unique_co=0:
      * auto/int Category -> OOF Target Encoding;
      * Category с encoding_type='freq' -> Frequency Encoding;
      * numeric -> обычная numeric-ветка LinearFeatures.

    output_categories=True не запускает OHEEncoder в финальной sparse/category-ветке.
    """
    return LinearFeatures(
        top_intersections=0,
        max_intersection_depth= 0,
        auto_unique_co=10,
        output_categories=True,
    )

## 7. Кастомный `StatsModelsGLM`

- fold-local `LinearFeatures` строится внутри `fit_predict_single_fold`;
- inner OOF TE group-safe;
- outer-valid трансформируется encoder-ом, который fit-ился только на outer-train;
- encoder сохраняется вместе с каждой fold-моделью и применяется к новым данным в `predict_single_fold`;
- `alpha` задаётся вектором, и для intercept `alpha[0] = 0`;
- feature cache позволяет не пересчитывать один и тот же Target Encoding на каждом trial Optuna.

При 29–30 итоговых признаках кэш обычно существенно ускоряет tuning, но потребляет дополнительную RAM. При дефиците памяти `cache_features=False`.

In [18]:
class StatsModelsGLM(TabularMLAlgo):
    _name = "SMGLM"

    _default_params = {
        "family": None,
        "alpha": 0.0,
        "L1_wt": 0.0,
        "method": "elastic_net",
        "maxiter": 2000,
    }

    # class-level cache переживает deepcopy в Optuna.
    _warmup_cache = {}
    _feature_cache = {}

    def __init__(
        self,
        *args,
        checkpoint_dir=None,
        inner_n_folds=N_INNER_FOLDS,
        inner_random_state=RANDOM_STATE,
        cache_features=True,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.checkpoint_dir = checkpoint_dir
        self.inner_n_folds = inner_n_folds
        self.inner_random_state = inner_random_state
        self.cache_features = cache_features
        self._fold_counter = 0

        if self.checkpoint_dir is not None:
            os.makedirs(self.checkpoint_dir, exist_ok=True)

    @classmethod
    def clear_caches(cls):
        cls._warmup_cache.clear()
        cls._feature_cache.clear()

    def _get_default_search_spaces(self, suggested_params, estimated_n_trials):
        return {
            "alpha": Uniform(low=0.1, high=1.0, log=True),
            "L1_wt": Uniform(low=0.0, high=0.3),
        }

    def _get_family(self):
        if self.task.name == "binary":
            return sm.families.Binomial()
        elif self.task.name == "reg":
            return sm.families.Poisson()
        raise ValueError(f"Task {self.task.name} is not supported by GLM wrapper")

    @staticmethod
    def _dataset_to_matrix(dataset, dtype=np.float32):
        arr = np.asarray(dataset.to_numpy().data, dtype=dtype)

        if arr.ndim != 2:
            raise ValueError(f"Ожидалась 2D-матрица, получено shape={arr.shape}")

        if not np.isfinite(arr).all():
            bad = np.argwhere(~np.isfinite(arr))
            raise ValueError(
                "После LinearFeatures остались NaN/inf. "
                f"Первые индексы: {bad[:10].tolist()}"
            )

        return arr

    def _prepare_fold_features(self, train, valid, fold_key):
        cache_key = int(fold_key)

        if self.cache_features and cache_key in StatsModelsGLM._feature_cache:
            return StatsModelsGLM._feature_cache[cache_key]

        if train.group is None:
            raise ValueError(
                "dataset.group отсутствует. GROUP должен иметь role='group', "
                "а не role='drop'."
            )

        inner_folds = make_group_safe_fold_ids(
            y=train.target,
            groups=train.group,
            n_splits=self.inner_n_folds,
            random_state=self.inner_random_state + cache_key,
        )

        train_for_feats = attach_folds(train, inner_folds)
        feats_local = make_linear_features()

        # OOF-признаки только внутри outer-train.
        transformed_train = feats_local.fit_transform(train_for_feats)

        # Ни target, ни statistics outer-valid не участвуют в fit encoder-а.
        transformed_valid = feats_local.transform(valid)

        X_train = self._dataset_to_matrix(transformed_train, dtype=np.float32)
        X_valid = self._dataset_to_matrix(transformed_valid, dtype=np.float32)
        feature_names = list(transformed_train.features)

        # При текущей конфигурации не должно появиться OHE-столбцов.
        ohe_features = [x for x in feature_names if "ohe_" in x.lower()]
        if ohe_features:
            raise RuntimeError(
                "Обнаружены OHE-признаки, хотя OHE запрещён: "
                + ", ".join(ohe_features[:10])
            )

        prepared = {
            "features_pipeline": feats_local,
            "X_train": X_train,
            "X_valid": X_valid,
            "feature_names": feature_names,
            "inner_folds": inner_folds,
        }

        if self.cache_features:
            StatsModelsGLM._feature_cache[cache_key] = prepared

        return prepared

    def fit_predict_single_fold(self, train, valid):
        family = self.params.get("family") or self._get_family()
        fold_key = self._fold_counter

        prepared = self._prepare_fold_features(train, valid, fold_key)

        # В cache оставляем float32, в statsmodels отдаём float64.
        X_train = sm.add_constant(
            np.asarray(prepared["X_train"], dtype=np.float64),
            has_constant="add",
        )
        X_valid = sm.add_constant(
            np.asarray(prepared["X_valid"], dtype=np.float64),
            has_constant="add",
        )
        y_train = np.asarray(train.target, dtype=np.float64).reshape(-1)

        # Warm start: unregularized GLM, один раз для каждого outer fold.
        if fold_key not in StatsModelsGLM._warmup_cache:
            try:
                model_unreg = sm.GLM(y_train, X_train, family=family)
                res_unreg = model_unreg.fit(
                    maxiter=min(int(self.params.get("maxiter", 2000)), 300),
                    disp=0,
                )
                start_params = np.asarray(res_unreg.params, dtype=np.float64)
            except Exception as exc:
                warnings.warn(
                    f"Unregularized warm start failed on fold={fold_key}: {exc}. "
                    "Используем intercept-only старт."
                )
                start_params = np.zeros(X_train.shape[1], dtype=np.float64)
                p = np.clip(y_train.mean(), 1e-8, 1 - 1e-8)
                start_params[0] = np.log(p / (1 - p))

            StatsModelsGLM._warmup_cache[fold_key] = start_params

        start_params = StatsModelsGLM._warmup_cache[fold_key]

        model = sm.GLM(y_train, X_train, family=family)

        # Критично: intercept не штрафуем.
        alpha_scalar = float(self.params["alpha"])
        alpha_vec = np.full(X_train.shape[1], alpha_scalar, dtype=np.float64)
        alpha_vec[0] = 0.0

        res = model.fit_regularized(
            method=self.params["method"],
            alpha=alpha_vec,
            L1_wt=float(self.params["L1_wt"]),
            start_params=start_params,
            maxiter=int(self.params.get("maxiter", 2000)),
        )

        val_pred = np.asarray(res.predict(X_valid), dtype=np.float64)

        # В каждой fold-модели сохраняем именно тот encoder, который был fitted
        # только на train этого outer fold.
        fold_model = {
            "glm": res,
            "features_pipeline": prepared["features_pipeline"],
            "feature_names": prepared["feature_names"],
            "fold_key": fold_key,
        }

        if self.checkpoint_dir is not None:
            trial_tag = getattr(self, "_trial_number", "final")
            path = os.path.join(
                self.checkpoint_dir,
                f"fold{self._fold_counter}_trial{trial_tag}.pkl",
            )
            joblib.dump(
                {"model": fold_model, "params": dict(self.params)},
                path,
            )

        self._fold_counter += 1
        return fold_model, val_pred

    def predict_single_fold(self, model, dataset):
        transformed = model["features_pipeline"].transform(dataset)
        X = self._dataset_to_matrix(transformed, dtype=np.float64)
        X = sm.add_constant(X, has_constant="add")
        return np.asarray(model["glm"].predict(X), dtype=np.float64)

## 8. Optuna tuner

По умолчанию `OptunaTuner` тюнится на holdout-версии первого outer fold, а после выбора параметров LAMA делает итоговый fit на полном `cv_iter`. Кэш fold 0 можно безопасно переиспользовать, потому что этот holdout соответствует первому outer split.

In [19]:
class CheckpointingOptunaTuner(OptunaTuner):
    def _get_objective(self, ml_algo, estimated_n_trials, train_valid_iterator):
        assert isinstance(ml_algo, MLAlgo)

        def objective(trial):
            _ml_algo = deepcopy(ml_algo)
            _ml_algo._trial_number = trial.number

            optimization_search_space = _ml_algo.optimization_search_space
            if not optimization_search_space:
                optimization_search_space = _ml_algo._get_default_search_spaces(
                    suggested_params=_ml_algo.init_params_on_input(train_valid_iterator),
                    estimated_n_trials=estimated_n_trials,
                )

            _ml_algo.params = self._sample(
                trial=trial,
                optimization_search_space=optimization_search_space,
                suggested_params=_ml_algo.init_params_on_input(train_valid_iterator),
            )

            output_dataset = _ml_algo.fit_predict(
                train_valid_iterator=train_valid_iterator
            )
            return _ml_algo.score(output_dataset)

        return objective

## 9. Reader и `MLPipeline`

Самое важное здесь: `feats = None`.

In [20]:
reader = PandasToPandasReader(
    task,
    cv=None,
    random_state=RANDOM_STATE,
    advanced_roles=False,
)

# Нельзя ставить сюда LinearFeatures(...): supervised feature engineering
# выполняется fold-local внутри StatsModelsGLM.
feats = None

tuner = CheckpointingOptunaTuner(
    n_trials=N_TRIALS,
    timeout=TIMEOUT,
)

model = StatsModelsGLM(
    checkpoint_dir=CHECKPOINT_DIR,
    inner_n_folds=N_INNER_FOLDS,
    inner_random_state=RANDOM_STATE,
    cache_features=True,
)

pipeline = MLPipeline(
    [(model, tuner)],
    pre_selection=None,
    features_pipeline=feats,
    post_selection=None,
)

automl = AutoML(
    reader,
    [[pipeline]],
    skip_conn=False,
)

## 10. Feature importance для `ReportDeco` — опционально

Permutation importance вызывается по raw features и использует полный `automl.predict`, поэтому fold-local preprocessing внутри моделей сохраняется.

In [21]:
if calc_feats_permutation_imps is not None:
    def get_feature_scores(
        self,
        calc_method="accurate",
        data=None,
        features_names=None,
        silent=False,
    ):
        if calc_method != "accurate" or data is None:
            if not silent:
                print(
                    "Fast/none feature importance not implemented for StatsModelsGLM "
                    "-- skipping FI section"
                )
            return None

        used_feats = self.reader.used_features
        return calc_feats_permutation_imps(
            self,
            used_feats,
            data,
            self.reader.target,
            self.reader.task.get_dataset_metric(),
            silent=silent,
        )

    automl.get_feature_scores = types.MethodType(get_feature_scores, automl)
else:
    print("calc_feats_permutation_imps не найден в этой версии LAMA — FI patch пропущен.")

## 11. Обучение

На каждом outer fold модель строит свой `LinearFeatures` и свой GLM. После CV LAMA хранит несколько fold-моделей и на новых данных усредняет их predictions.

In [22]:
from matplotlib.axes import Axes

_original_boxplot = Axes.boxplot
def _patched_boxplot(self, *args, **kwargs):
    if "labels" in kwargs and "tick_labels" not in kwargs:
        kwargs["tick_labels"] = kwargs.pop("labels")
    return _original_boxplot(self, *args, **kwargs)
Axes.boxplot = _patched_boxplot

In [23]:
StatsModelsGLM.clear_caches()

report_automl = ReportDeco(
    output_path="lama_report/"
)(automl)

oof_pred = report_automl.fit_predict(
    train_data,
    roles=roles,
    cv_iter=cv_iter,
    path_to_save="model.joblib",
    verbose=4,
)

[11:37:18] Train data shape: (405634, 39)

[11:37:18] Layer 1 train process start. Time left 9999999999.53 secs
[11:37:19] Start hyperparameters optimization for Lvl_0_Pipe_0_Mod_0_SMGLM ... Time budget is 21600.00 secs
[11:37:19] Copying TaskTimer may affect the parent PipelineTimer, so copy will create new unlimited TaskTimer
[11:38:14] Lvl_0_Pipe_0_Mod_0_SMGLM fitting and predicting completed
[11:38:14] Trial 1 with hyperparameters {'alpha': 0.23688639503640782, 'L1_wt': 0.2852142919229748} scored 0.7399026304644127 in 0:00:55.080919
[11:38:42] Lvl_0_Pipe_0_Mod_0_SMGLM fitting and predicting completed
[11:38:42] Trial 2 with hyperparameters {'alpha': 0.5395030966670228, 'L1_wt': 0.17959754525911098} scored 0.5 in 0:00:28.135503
[11:45:26] Lvl_0_Pipe_0_Mod_0_SMGLM fitting and predicting completed
[11:45:26] Trial 3 with hyperparameters {'alpha': 0.1432249371823025, 'L1_wt': 0.04679835610086079} scored 0.7930872427668073 in 0:06:43.183966
[11:46:30] Lvl_0_Pipe_0_Mod_0_SMGLM fitting an

In [24]:
fitted_algo = automl.levels[0][0].ml_algos[0]
fold_coef_tables = []
for fold, fold_model in enumerate(fitted_algo.models):
    res = fold_model["glm"]
    # Первый коэффициент — intercept
    coef = np.asarray(res.params)[1:]
    feature_names = list(fold_model["feature_names"])
    fold_df = pd.DataFrame({
        "feature": feature_names,
        "coef": coef,
        "abs_coef": np.abs(coef),
        "fold": fold,
    })
    fold_df["is_zero"] = fold_df["abs_coef"] < 1e-8
    fold_coef_tables.append(fold_df)
coef_df = pd.concat(
    fold_coef_tables,
    ignore_index=True
)
coef_df.head()


,feature,coef,abs_coef,fold,is_zero
0,scaler__fillnamed__fillinf__CNT_CLINIC_IN_PROGRAM,-0.120275,0.120275,0,False
1,scaler__fillnamed__fillinf__CNT_PROGRAM/cnt_po...,0.217437,0.217437,0,False
2,scaler__fillnamed__fillinf__COMISSION_PERCENTA...,0.011358,0.011358,0,False
3,scaler__fillnamed__fillinf__COVENANTA_ZHUD,0.000000,0.000000,0,True
4,scaler__fillnamed__fillinf__DAYS,0.037841,0.037841,0,False


In [25]:
zero_stats = (
    coef_df
    .groupby("fold")
    .agg(
        n_features=("feature", "size"),
        n_zero=("is_zero", "sum"),
    )
)
zero_stats["n_nonzero"] = (
    zero_stats["n_features"]
    - zero_stats["n_zero"]
)
zero_stats["zero_share"] = (
    zero_stats["n_zero"]
    / zero_stats["n_features"]
)
zero_stats


,n_features,n_zero,n_nonzero,zero_share
fold,,,,
0,37,16,21,0.432432
1,37,16,21,0.432432
2,37,18,19,0.486486
3,37,19,18,0.513514


In [26]:
coef_wide = (
    coef_df
    .pivot(
        index="feature",
        columns="fold",
        values="coef"
    )
)
nonzero = coef_wide.abs() >= 1e-8
stability = pd.DataFrame({
    "n_folds_nonzero": nonzero.sum(axis=1),
    "share_folds_nonzero": nonzero.mean(axis=1),
    "mean_abs_coef": coef_wide.abs().mean(axis=1),
    "mean_coef": coef_wide.mean(axis=1),
})
stability = stability.sort_values(
    ["n_folds_nonzero", "mean_abs_coef"],
    ascending=False
)
stability


,n_folds_nonzero,share_folds_nonzero,mean_abs_coef,mean_coef
feature,,,,
scaler__fillnamed__fillinf__CNT_PROGRAM/cnt_policy_beg,4,1.00,0.216613,0.216613
scaler__fillnamed__fillinf__logodds__oof__le__INSURANT_NAME,4,1.00,0.130706,0.130706
scaler__fillnamed__fillinf__CNT_CLINIC_IN_PROGRAM,4,1.00,0.117495,-0.117495
scaler__fillnamed__fillinf__SHARE_LPU_IN_PROGRAM,4,1.00,0.089957,0.089957
scaler__fillnamed__fillinf__SHARE_PRICE_LPU_FROM_MAX_sq,4,1.00,0.073108,0.073108
scaler__fillnamed__fillinf__PRICE_CLINIC_RANK,4,1.00,0.050102,-0.050102
scaler__fillnamed__fillinf__DAYS,4,1.00,0.038559,0.038559
scaler__fillnamed__fillinf__diff_DATE_START_RESP,4,1.00,0.034772,-0.034772
scaler__fillnamed__fillinf__logodds__oof__le__INSURANT_REGION,4,1.00,0.033702,0.033702


## 12. OOF diagnostics

Точное равенство `mean(OOF prediction) == mean(y)` не является обязательным: каждая строка предсказывается моделью, которая не обучалась на этой строке и её группе. Но большой систематический gap требует проверки калибровки.

In [26]:
def _compare_feature_score(self, *args, **kwargs):
    return None

AutoML.get_feature_scores = _compare_feature_score

report_automl = joblib.load('model.joblib') 

In [27]:
fitted_algo = report_automl.levels[0][0].ml_algos[0]
print(type(fitted_algo))
print("Количество fold models:", len(fitted_algo.models))

<class '__main__.StatsModelsGLM'>
Количество fold models: 4


In [29]:
train_data = report_automl.reader.read(train_data, add_array_attrs=True)

In [30]:
oof_p = np.full(
    len(train_data),
    np.nan,
    dtype=float,
)
for fold, (_, valid_idx) in enumerate(cv_iter):
    fold_model = fitted_algo.models[fold]
    valid_dataset = train_data[valid_idx]
    fold_pred = fitted_algo.predict_single_fold(
        fold_model,
        valid_dataset,
    )
    oof_p[valid_idx] = np.asarray(fold_pred).ravel()
assert np.isfinite(oof_p).all()
print("OOF shape:", oof_p.shape)
print("Target mean:", train_data[TARGET_NAME].mean())
print("OOF mean:", oof_p.mean())


AssertionError: 

In [35]:
print("NaN:", np.isnan(oof_p).sum())
print("All:", len(oof_p))
print("Inf:", np.isinf(oof_p).sum())

NaN: 101410
All: 405633
Inf: 0


In [37]:
for fold, (_, valid_isx) in enumerate(cv_iter):
    valid_dataset = train_data[valid_idx]
    fold_pred = fitted_algo.predict_single_fold(
        fitted_algo.models[fold],
        valid_dataset,
    )

    p = np.asarray(fold_pred).ravel()
    print(
        f'fold {fold}:',
        f'n={len(p)}',
        f'nan={np.isnan(p).sum()}',
        f'inf={np.isinf(p).sum()}',
        f'min={np.nanmin(p)}',
        f'max={np.nanmax(p)}'
    )

fold 0: n=101407 nan=0 inf=0 min=0.013005296921717813 max=0.6763654363793449


C:\Temp\Kirill.Vinogradov\24\ipykernel_90760\3838493659.py:14: RuntimeWarning: All-NaN slice encountered
  f'min={np.nanmin(p)}',
C:\Temp\Kirill.Vinogradov\24\ipykernel_90760\3838493659.py:15: RuntimeWarning: All-NaN slice encountered
  f'max={np.nanmax(p)}'


fold 1: n=101407 nan=101407 inf=0 min=nan max=nan
fold 2: n=101407 nan=0 inf=0 min=0.013669424698351775 max=0.9188107956464489
fold 3: n=101407 nan=0 inf=0 min=0.013049560342509658 max=0.8533410771476381


In [41]:
bad_fold = 1

fold_model = fitted_algo.models[bad_fold]
res = fold_model['glm']

params = np.asarray(res.params)

print("params:", params.shape)
print('nan:', np.isnan(params).sum())

params: (40,)
nan: 40


In [42]:
train_idx, val_idx = cv_iter[bad_fold]

train_data_bad = train_data[train_idx]

X_train_bad = (
    fold_model['features_pipeline']
    .transform(train_data_bad)
    .to_pandas()
    .data
    .astype('float64')
)

print(
    'X finite:',
    np.isfinite(X_train_bad.to_numpy()).all()
)

X finite: True


In [45]:
y_all = train_data.target

y_bad = y_all[train_idx]

print(y_bad.value_counts())

PAID_FLAG_1
0    283180
1     21043
Name: count, dtype: int64


In [46]:
X_bad_sm = sm.add_constant(
    X_train_bad.to_numpy(),
    has_constant='add'
)

model_unreg = sm.GLM(
    y_bad.to_numpy(),
    X_bad_sm,
    family=sm.families.Binomial()
)

res_unreg = model_unreg.fit()

print('Unreg params finite:', np.isfinite(res_unreg.params).all())

print('max |beta|', np.nanmax(np.abs(res_unreg.params)))

Unreg params finite: True
max |beta| 2.8382593419584956


In [55]:
def cond_check(data, feature_names):
    X = np.asarray(data, dtype=float)

    bad_cond = np.linalg.cond(X)

    rows = []

    for j, feature in enumerate(feature_names):

        X_red = np.delete(
            X,
            j,
            axis=1
        )

        cond = np.linalg.cond(X_red)

        rows.append({
            'feature': feature,
            'cond_full': bad_cond,
            'cond_without': cond,
            'improv_ratio': (
                bad_cond / cond
                if np.isfinite(cond) and cond > 0
                else np.nan
            ),
        })

    return (
            pd.DataFrame(rows)
            .sort_values(
                'improv_ratio',
                ascending=False
            )
        )

In [56]:
cond_check(X_train_bad, X_train_bad.columns)

,feature,cond_full,cond_without,improv_ratio
35,le__INSURANT_REGION_GROUP,36.274233,23.214303,1.562581
37,le__PROLONGATION_TYPE,36.274233,32.430253,1.118531
36,le__PROGRAM_LIST_RISK_PREPAY_SPEC,36.274233,34.044481,1.065495
34,nanflg__DOGOVOR_TYPE,36.274233,34.375961,1.055221
38,le__Pol,36.274233,34.849294,1.040889
27,scaler__fillnamed__fillinf__diff_DATE_START_RESP,36.274233,35.708161,1.015853
23,scaler__fillnamed__fillinf__UNPINNING,36.274233,36.084617,1.005255
4,scaler__fillnamed__fillinf__DAYS,36.274233,36.092023,1.005048
5,scaler__fillnamed__fillinf__DAYS_sq,36.274233,36.104374,1.004705
28,scaler__fillnamed__fillinf__diff_RESP_END_START,36.274233,36.180137,1.002601


In [27]:
def pred_to_1d(pred_dataset):
    arr = np.asarray(pred_dataset.data)
    if arr.ndim == 2:
        arr = arr[:, 0]
    return arr.astype(float)


oof_p = pred_to_1d(oof_pred)
oof_y = train_data[TARGET_NAME].to_numpy(dtype=float)

mask = np.isfinite(oof_p)

print("OOF rows   :", mask.sum(), "/", len(mask))
print("mean(y)    :", oof_y[mask].mean())
print("mean(pred) :", oof_p[mask].mean())
print("mean gap   :", oof_p[mask].mean() - oof_y[mask].mean())
print("ROC-AUC    :", roc_auc_score(oof_y[mask], oof_p[mask]))
print(
    "LogLoss    :",
    log_loss(oof_y[mask], np.clip(oof_p[mask], 1e-12, 1 - 1e-12)),
)
print("Brier      :", brier_score_loss(oof_y[mask], oof_p[mask]))

OOF rows   : 405634 / 405634
mean(y)    : 0.06921017468949842
mean(pred) : 0.07379917947366811
mean gap   : 0.004589004784169695
ROC-AUC    : 0.800257594012262
LogLoss    : 0.22789604390511534
Brier      : 0.06013304415943686


OOF rows   : 405634 / 405634  
mean(y)    : 0.06921017468949842  
mean(pred) : 0.07381354286453629  
mean gap   : 0.00460336817503787  
ROC-AUC    : 0.7999948848163553  
LogLoss    : 0.22811593441404543  
Brier      : 0.06016739317841876  

## 13. Holdout prediction

В `test_data` сохраняем `GROUP_NAME`: reader использует его как metadata, но он не становится predictor-ом.

In [28]:
# test_features = test_data.drop(columns=[TARGET_NAME])

holdout_pred = report_automl.predict(test_data)
holdout_p = pred_to_1d(holdout_pred)
holdout_y = test_data[TARGET_NAME].to_numpy(dtype=float)

print("mean(y)    :", holdout_y.mean())
print("mean(pred) :", holdout_p.mean())
print("mean gap   :", holdout_p.mean() - holdout_y.mean())
print("ROC-AUC    :", roc_auc_score(holdout_y, holdout_p))
print(
    "LogLoss    :",
    log_loss(holdout_y, np.clip(holdout_p, 1e-12, 1 - 1e-12)),
)
print("Brier      :", brier_score_loss(holdout_y, holdout_p))

mean(y)    : 0.06920353807772332
mean(pred) : 0.06887758403540856
mean gap   : -0.0003259540423147661
ROC-AUC    : 0.8062414164663525
LogLoss    : 0.21915636523470405
Brier      : 0.058732851168695276


## 14. Доверительный интервал для среднего с учётом `GROUP`

При внутригрупповой зависимости нельзя считать все ~700k строк независимыми для CI. Ниже bootstrap ресемплирует **целые группы**, сохраняя все строки выбранной группы.

Это CI для агрегированной калибровки/среднего, а не Wald CI для elastic-net коэффициентов.

In [29]:
def group_bootstrap_mean_gap(
    y,
    pred,
    groups,
    n_boot=2000,
    random_state=RANDOM_STATE,
):
    df = pd.DataFrame({
        "group": np.asarray(groups),
        "y": np.asarray(y, dtype=float),
        "pred": np.asarray(pred, dtype=float),
    })

    agg = (
        df.groupby("group", dropna=False)
        .agg(
            n=("y", "size"),
            sum_y=("y", "sum"),
            sum_pred=("pred", "sum"),
        )
        .reset_index(drop=True)
    )

    rng = np.random.default_rng(random_state)
    m = len(agg)

    n_arr = agg["n"].to_numpy(float)
    sy = agg["sum_y"].to_numpy(float)
    sp = agg["sum_pred"].to_numpy(float)

    obs_means = np.empty(n_boot, dtype=float)
    pred_means = np.empty(n_boot, dtype=float)
    gaps = np.empty(n_boot, dtype=float)

    for b in range(n_boot):
        idx = rng.integers(0, m, size=m)
        denom = n_arr[idx].sum()
        obs = sy[idx].sum() / denom
        prd = sp[idx].sum() / denom

        obs_means[b] = obs
        pred_means[b] = prd
        gaps[b] = prd - obs

    return {
        "observed_mean": float(np.mean(y)),
        "predicted_mean": float(np.mean(pred)),
        "gap": float(np.mean(pred) - np.mean(y)),
        "observed_ci95": tuple(np.quantile(obs_means, [0.025, 0.975])),
        "predicted_ci95": tuple(np.quantile(pred_means, [0.025, 0.975])),
        "gap_ci95": tuple(np.quantile(gaps, [0.025, 0.975])),
    }


holdout_bootstrap = group_bootstrap_mean_gap(
    y=holdout_y,
    pred=holdout_p,
    groups=test_data[GROUP_NAME].to_numpy(),
    n_boot=2000,
)

holdout_bootstrap

{'observed_mean': 0.06920353807772332,
 'predicted_mean': 0.06887758403540856,
 'gap': -0.0003259540423147661,
 'observed_ci95': (0.05877821801609051, 0.08047220834819103),
 'predicted_ci95': (0.06585179017323198, 0.07191887276379345),
 'gap_ci95': (-0.009496754980781019, 0.008182892682844242)}

## 15. Диагностика фактического входа GLM

После исправления больше нет одного глобального `X_model_view`, потому что каждый outer fold имеет свой encoder. Для отладки ниже строится `X_model_view` только для **первого outer-train** по той же схеме, что использует модель.

In [30]:
reader_diag = PandasToPandasReader(
    task,
    cv=None,
    random_state=RANDOM_STATE,
    advanced_roles=False,
)

train_dataset = reader_diag.fit_read(
    train_data,
    roles=roles,
)

print("GROUP в model features:", GROUP_NAME in train_dataset.features)
print("dataset.group exists  :", train_dataset.group is not None)

assert GROUP_NAME not in train_dataset.features
assert train_dataset.group is not None

tr_idx, va_idx = cv_iter[0]
outer_train_ds = train_dataset[tr_idx]
outer_valid_ds = train_dataset[va_idx]

assert len(set(outer_train_ds.group) & set(outer_valid_ds.group)) == 0

inner_folds = make_group_safe_fold_ids(
    outer_train_ds.target,
    outer_train_ds.group,
    n_splits=N_INNER_FOLDS,
    random_state=RANDOM_STATE,
)

outer_train_for_feats = attach_folds(
    outer_train_ds,
    inner_folds,
)

diag_feats = make_linear_features()
transformed_train = diag_feats.fit_transform(outer_train_for_feats)
transformed_valid = diag_feats.transform(outer_valid_ds)

X_model_view = transformed_train.to_pandas().data

print("X_model_view shape:", X_model_view.shape)
print("Дубли transformed outer-train:")
print(duplicate_stats(X_model_view))

print("\nПервые имена признаков:")
print(pd.Series(X_model_view.columns).head(60).to_string(index=False))

[12:20:37] Train data shape: (405634, 39)

GROUP в model features: False
dataset.group exists  : True
X_model_view shape: (304227, 37)
Дубли transformed outer-train:
{'rows': 304227, 'unique_patterns': 300815, 'extra_duplicates': 3412, 'rows_in_duplicate_groups': 6824, 'duplicate_groups': 3412, 'max_duplicate_group_size': 2}

Первые имена признаков:
 scaler__fillnamed__fillinf__CNT_CLINIC_IN_PROGRAM
scaler__fillnamed__fillinf__CNT_PROGRAM/cnt_pol...
scaler__fillnamed__fillinf__COMISSION_PERCENTAG...
        scaler__fillnamed__fillinf__COVENANTA_ZHUD
                  scaler__fillnamed__fillinf__DAYS
        scaler__fillnamed__fillinf__DIRECT_SERVICE
          scaler__fillnamed__fillinf__DOGOVOR_TYPE
            scaler__fillnamed__fillinf__GROUP_SIZE
       scaler__fillnamed__fillinf__INSURED_EXP_NEW
               scaler__fillnamed__fillinf__INS_AGE
           scaler__fillnamed__fillinf__MAX_REVIEWS
             scaler__fillnamed__fillinf__MONTH_SIN
     scaler__fillnamed__fillinf__PRI

## 16. Как читать диагностику дублей по колонкам

Большой `delta` после удаления признака означает: **этот признак хорошо различает строки**. Это не означает, что он «создаёт дубли».

In [31]:
def diagnose_duplicate_sources(df, sample_frac=None):
    if sample_frac is not None and sample_frac < 1.0:
        df = df.sample(
            frac=sample_frac,
            random_state=RANDOM_STATE,
        ).reset_index(drop=True)

    base_dupes = df.duplicated(keep=False).sum()
    print(f"Базовое число строк в duplicate groups: {base_dupes}")

    results = []
    cols = df.columns.tolist()

    for col in cols:
        subset = [c for c in cols if c != col]
        dupes_without_col = df.duplicated(
            subset=subset,
            keep=False,
        ).sum()

        results.append({
            "column": col,
            "unique_values": int(df[col].nunique(dropna=False)),
            "dupes_without_col": int(dupes_without_col),
            "delta": int(dupes_without_col - base_dupes),
        })

    return (
        pd.DataFrame(results)
        .sort_values("delta", ascending=False)
        .reset_index(drop=True)
    )


diag = diagnose_duplicate_sources(X_model_view)
diag.head(30)

Базовое число строк в duplicate groups: 6824


,column,unique_values,dupes_without_col,delta
0,scaler__fillnamed__fillinf__INS_AGE,64,248689,241865
1,le__Pol,2,65730,58906
2,scaler__fillnamed__fillinf__GROUP_SIZE,553,24854,18030
3,scaler__fillnamed__fillinf__PRICE_CLINIC_RANK,44,23231,16407
4,scaler__fillnamed__fillinf__VIP,2,10642,3818
5,scaler__fillnamed__fillinf__INSURED_EXP_NEW,167,7297,473
6,nanflg__DOGOVOR_TYPE,2,7285,461
7,scaler__fillnamed__fillinf__DIRECT_SERVICE,2,7087,263
8,scaler__fillnamed__fillinf__diff_DATE_START_RESP,345,7022,198
9,scaler__fillnamed__fillinf__DOGOVOR_TYPE,3,6888,64


## 17. Дополнительная проверка: одинаковые X и разные y

Это полезнее, чем просто число дублей. Для каждой группы полностью одинаковых transformed-X можно посмотреть размер, target rate и разброс target. GLM закономерно выдаёт одинаковую вероятность всем строкам одной такой группы.

In [32]:
def duplicate_pattern_target_summary(X_df, y, top_n=20):
    tmp = X_df.copy()
    tmp["__target__"] = np.asarray(y)

    feature_cols = list(X_df.columns)
    out = (
        tmp.groupby(feature_cols, dropna=False, sort=False)["__target__"]
        .agg(["size", "mean", "sum"])
        .query("size > 1")
        .sort_values("size", ascending=False)
        .head(top_n)
        .reset_index()
    )
    return out


dup_patterns = duplicate_pattern_target_summary(
    X_model_view,
    outer_train_ds.target,
    top_n=20,
)

dup_patterns[["size", "mean", "sum"]]

,size,mean,sum
0,2,0.5,1
1,2,0.5,1
2,2,0.5,1
3,2,0.5,1
4,2,0.5,1
5,2,0.5,1
6,2,0.5,1
7,2,0.5,1
8,2,0.5,1
9,2,0.5,1


## 18. Проверка fitted моделей после обучения

Каждый элемент `models` теперь является bundle: GLM + соответствующий fold-local `LinearFeatures`.

In [ ]:
# Структура levels может немного отличаться между версиями LAMA.
# Сначала удобно посмотреть объект:
print(report_automl)

# В большинстве версий после fit модель можно получить через automl.levels.
# Если путь в вашей версии отличается, достаточно найти StatsModelsGLM и проверить:
#   len(fitted_algo.models)
#   fitted_algo.models[0].keys()
# Ожидаемые keys: glm, features_pipeline, feature_names, fold_key.


## 19. Финальная сборка `X_train_view`, `X_test_view`, `full`

В новой group-safe архитектуре глобального `automl.levels[0][0].features_pipeline` больше нет: у каждой outer-fold модели свой `LinearFeatures`.

Поэтому для единого экспортного пространства признаков создаём отдельный **final/report encoder**:

- он обучается на всём `train_data`;
- для `X_train_view` использует group-safe OOF Target Encoding через `fit_transform`;
- для `X_test_view` использует статистики, fitted на всём `train_data`, через `transform`;
- OHE не используется;
- колонки train/test гарантированно совпадают.

Важно: `oof_pred` остаётся честным OOF-прогнозом outer-CV, а `test_pred` — прогнозом ансамбля outer-fold моделей.  
`X_train_view`/`X_test_view` ниже нужны для EVA, анализа, экспорта и проверки признаков; это единое представление, а не буквальная матрица каждого отдельного outer-fold GLM.


In [33]:

# 19.1. Предсказания для train (OOF) и независимого holdout

oof_p = pred_to_1d(oof_pred)

# Для predict target не нужен.
# test_features = test_data.drop(columns=[TARGET_NAME], errors="ignore").copy()
test_pred = report_automl.predict(test_data)
test_p = pred_to_1d(test_pred)

assert len(oof_p) == len(train_data)
assert len(test_p) == len(test_data)

print("OOF predictions :", oof_p.shape)
print("Test predictions:", test_p.shape)


OOF predictions : (405634,)
Test predictions: (101411,)


In [34]:

# 19.2. Строим отдельный reader для единого final/report feature space

reader_final = PandasToPandasReader(
    task,
    cv=None,
    random_state=RANDOM_STATE,
    advanced_roles=False,
)

train_dataset_final = reader_final.fit_read(
    train_data,
    roles=roles,
)

# GROUP хранится как special array attribute и не является model feature.
assert train_dataset_final.group is not None
assert GROUP_NAME not in train_dataset_final.features

# Для train-части final encoder создаём group-safe OOF folds.
final_inner_folds = make_group_safe_fold_ids(
    y=train_dataset_final.target,
    groups=train_dataset_final.group,
    n_splits=N_INNER_FOLDS,
    random_state=RANDOM_STATE + 100_000,
)

train_dataset_final_oof = attach_folds(
    train_dataset_final,
    final_inner_folds,
)

feats_final = make_linear_features()

# OOF TE для train:
transformed_train_final = feats_final.fit_transform(
    train_dataset_final_oof
)

# Для test нужен dataset в тех же ролях/feature definitions.
# Target в test_data может присутствовать для оценки, но encoder его не использует.
test_dataset_final = reader_final.read(
    test_data,
    add_array_attrs=False,
)

# Полные train-статистики применяются к holdout:
transformed_test_final = feats_final.transform(
    test_dataset_final
)

X_train_view = (
    transformed_train_final
    .to_pandas()
    .data
    .reset_index(drop=True)
)

X_test_view = (
    transformed_test_final
    .to_pandas()
    .data
    .reset_index(drop=True)
)

assert list(X_train_view.columns) == list(X_test_view.columns)
assert len(X_train_view) == len(train_data)
assert len(X_test_view) == len(test_data)

print("X_train_view:", X_train_view.shape)
print("X_test_view :", X_test_view.shape)
print("Совпадают колонки:", list(X_train_view.columns) == list(X_test_view.columns))


[12:21:04] Train data shape: (405634, 39)

X_train_view: (405634, 37)
X_test_view : (101411, 37)
Совпадают колонки: True


In [35]:

# 19.3. Добавляем служебные поля и prediction

train_meta = train_data.reset_index(drop=True)
test_meta = test_data.reset_index(drop=True)

X_train_view = X_train_view.copy()
X_train_view["sample"] = "TRAIN"
X_train_view[TARGET_NAME] = train_meta[TARGET_NAME].to_numpy()
X_train_view["GROUP"] = train_meta[GROUP_NAME].to_numpy()

if START_DATE_NAME in train_meta.columns:
    X_train_view["START_DATE"] = train_meta[START_DATE_NAME].to_numpy()

X_train_view["prediction"] = oof_p


X_test_view = X_test_view.copy()
X_test_view["sample"] = "OOS"

if TARGET_NAME in test_meta.columns:
    X_test_view[TARGET_NAME] = test_meta[TARGET_NAME].to_numpy()

X_test_view["GROUP"] = test_meta[GROUP_NAME].to_numpy()

if START_DATE_NAME in test_meta.columns:
    X_test_view["START_DATE"] = test_meta[START_DATE_NAME].to_numpy()

X_test_view["prediction"] = test_p


full = pd.concat(
    [X_train_view, X_test_view],
    axis=0,
    ignore_index=True,
)

print("full:", full.shape)
full.head()


full: (507045, 42)


,scaler__fillnamed__fillinf__CNT_CLINIC_IN_PROGRAM,scaler__fillnamed__fillinf__CNT_PROGRAM/cnt_policy_beg,scaler__fillnamed__fillinf__COMISSION_PERCENTAGE_FIX,scaler__fillnamed__fillinf__COVENANTA_ZHUD,scaler__fillnamed__fillinf__DAYS,scaler__fillnamed__fillinf__DIRECT_SERVICE,scaler__fillnamed__fillinf__DOGOVOR_TYPE,scaler__fillnamed__fillinf__GROUP_SIZE,scaler__fillnamed__fillinf__INSURED_EXP_NEW,scaler__fillnamed__fillinf__INS_AGE,scaler__fillnamed__fillinf__MAX_REVIEWS,scaler__fillnamed__fillinf__MONTH_SIN,scaler__fillnamed__fillinf__PRICE_CLINIC_RANK,scaler__fillnamed__fillinf__PRICE_LEVEL_NUM,scaler__fillnamed__fillinf__PRICE_VALUE,scaler__fillnamed__fillinf__SHARE_BUS_DIST_MIN_FROM_MIN,scaler__fillnamed__fillinf__SHARE_LPU_IN_PROGRAM,scaler__fillnamed__fillinf__SHARE_PRICE_LPU_FROM_MAX,scaler__fillnamed__fillinf__SHARE_PRICE_LPU_FROM_MAX_sq,scaler__fillnamed__fillinf__SHARE_REVIEW_FROM_MAX,scaler__fillnamed__fillinf__TIME_TREND,scaler__fillnamed__fillinf__UNPINNING,scaler__fillnamed__fillinf__VIP,scaler__fillnamed__fillinf__WEEK_COS,scaler__fillnamed__fillinf__WEEK_SIN,scaler__fillnamed__fillinf__diff_DATE_START_RESP,scaler__fillnamed__fillinf__diff_RESP_END_START,scaler__fillnamed__fillinf__logodds__oof__le__INSURANT_NAME,scaler__fillnamed__fillinf__logodds__oof__le__INSURANT_REGION,scaler__fillnamed__fillinf__logodds__oof__le__OKVED_GROUP,scaler__fillnamed__fillinf__logodds__oof__le__PRICE_LEVEL,scaler__fillnamed__fillinf__logodds__oof__le__SHARE_SUBWAY_DIST_MIN_BINS,nanflg__DOGOVOR_TYPE,le__INSURANT_REGION_GROUP,le__Pol,le__PROGRAM_LIST_RISK_PREPAY_SPEC,le__PROLONGATION_TYPE,sample,PAID_FLAG_1,GROUP,START_DATE,prediction
0,-0.935176,0.143471,-0.222698,-0.500985,0.447039,0.249113,0.163344,-0.662781,0.459638,-0.145467,-0.450904,-0.122356,-0.480137,-0.896379,-0.187224,0.878105,0.18001,1.114663,0.917095,1.094726,-0.63716,-0.385277,-0.266609,0.911154,-1.012998,-0.464224,-0.087575,0.669571,0.544214,0.113901,0.241232,-0.136265,0.0,5.0,2.0,2.0,3.0,TRAIN,0,845/Klinika Ekspert Tver'/69-744-01,2023-10-09 00:00:00,0.121106
1,-0.935176,0.143471,-0.222698,-0.500985,0.447039,0.249113,0.163344,-0.662781,0.459638,-0.908655,-0.450904,-0.122356,-0.480137,-0.896379,-0.187224,0.878105,0.18001,1.114663,0.917095,1.094726,-0.63716,-0.385277,-0.266609,0.911154,-1.012998,-0.464224,-0.087575,0.669571,0.544214,0.113901,0.241232,-0.136265,0.0,5.0,2.0,2.0,3.0,TRAIN,0,845/Klinika Ekspert Tver'/69-744-01,2023-10-09 00:00:00,0.121106
2,-0.935176,0.143471,-0.222698,-0.500985,0.447039,0.249113,0.163344,-0.662781,0.459638,0.331526,-0.450904,-0.122356,-0.480137,-0.896379,-0.187224,0.878105,0.18001,1.114663,0.917095,1.094726,-0.63716,-0.385277,-0.266609,0.911154,-1.012998,-0.464224,-0.087575,0.669571,0.544214,0.113901,0.241232,-0.136265,0.0,5.0,2.0,2.0,3.0,TRAIN,0,845/Klinika Ekspert Tver'/69-744-01,2023-10-09 00:00:00,0.121106
3,-0.935176,0.143471,-0.222698,-0.500985,0.447039,0.249113,0.163344,-0.662781,0.459638,-0.240865,-0.450904,-0.122356,-0.480137,-0.896379,-0.187224,0.878105,0.18001,1.114663,0.917095,1.094726,-0.63716,-0.385277,-0.266609,0.911154,-1.012998,-0.464224,-0.087575,0.669571,0.544214,0.113901,0.241232,-0.136265,0.0,5.0,2.0,2.0,3.0,TRAIN,0,845/Klinika Ekspert Tver'/69-744-01,2023-10-09 00:00:00,0.121106
4,-0.935176,0.143471,-0.222698,-0.500985,0.447039,0.249113,0.163344,-0.662781,0.459638,0.713120,-0.450904,-0.122356,-0.480137,-0.896379,-0.187224,0.878105,0.18001,1.114663,0.917095,1.094726,-0.63716,-0.385277,-0.266609,0.911154,-1.012998,-0.464224,-0.087575,0.669571,0.544214,0.113901,0.241232,-0.136265,0.0,5.0,2.0,2.0,3.0,TRAIN,0,845/Klinika Ekspert Tver'/69-744-01,2023-10-09 00:00:00,0.121106


In [36]:

# 19.4. Контроль финальной сборки

print(full["sample"].value_counts(dropna=False))

print("\nTrain target / prediction mean:")
print("target     =", X_train_view[TARGET_NAME].mean())
print("prediction =", X_train_view["prediction"].mean())

if TARGET_NAME in X_test_view.columns:
    print("\nOOS target / prediction mean:")
    print("target     =", X_test_view[TARGET_NAME].mean())
    print("prediction =", X_test_view["prediction"].mean())

print("\nДубли в едином transformed feature space, без служебных колонок:")

service_cols = {
    "sample",
    TARGET_NAME,
    "GROUP",
    "START_DATE",
    "prediction",
}

model_cols = [
    c for c in full.columns
    if c not in service_cols
]

print("TRAIN:", duplicate_stats(X_train_view[model_cols]))
print("OOS  :", duplicate_stats(X_test_view[model_cols]))


sample
TRAIN    405634
OOS      101411
Name: count, dtype: int64

Train target / prediction mean:
target     = 0.06921017468949842
prediction = 0.07379917947366811

OOS target / prediction mean:
target     = 0.06920353807772332
prediction = 0.06887758403540856

Дубли в едином transformed feature space, без служебных колонок:
TRAIN: {'rows': 405634, 'unique_patterns': 401002, 'extra_duplicates': 4632, 'rows_in_duplicate_groups': 9264, 'duplicate_groups': 4632, 'max_duplicate_group_size': 2}
OOS  : {'rows': 101411, 'unique_patterns': 100262, 'extra_duplicates': 1149, 'rows_in_duplicate_groups': 2298, 'duplicate_groups': 1149, 'max_duplicate_group_size': 2}


In [ ]:

# 19.5. Сохраняем единый final/report encoder и при необходимости full

# FINAL_ENCODER_PATH = "final_report_linear_features.pkl"

# joblib.dump(
#     {
#         "reader": reader_final,
#         "features_pipeline": feats_final,
#         "feature_names": list(transformed_train_final.features),
#         "target_name": TARGET_NAME,
#         "group_name": GROUP_NAME,
#         "start_date_name": START_DATE_NAME,
#     },
#     FINAL_ENCODER_PATH,
# )

# # Parquet можно закомментировать, если не нужен файл на диске.
# file_path = os.path.join(os.getcwd(), 'bin_subsample_ACT.csv')
# full.to_csv(file_path, index=False)
# print(f"Файл сохранен по пути: {file_path}")

# print("Сохранён encoder:", FINAL_ENCODER_PATH)


Файл сохранен по пути: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Виноградов\bin_subsample_ACT.csv
Сохранён encoder: final_report_linear_features.pkl


In [ ]:
full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 509505 entries, 0 to 509504
Data columns (total 47 columns):
 #   Column                                                                    Non-Null Count   Dtype  
---  ------                                                                    --------------   -----  
 0   scaler__fillnamed__fillinf__CNT_CLINIC_IN_PROGRAM                         509505 non-null  float32
 1   scaler__fillnamed__fillinf__CNT_PROGRAM                                   509505 non-null  float32
 2   scaler__fillnamed__fillinf__CNT_PROGRAM/cnt_policy_beg                    509505 non-null  float32
 3   scaler__fillnamed__fillinf__COMISSION_PERCENTAGE_FIX                      509505 non-null  float32
 4   scaler__fillnamed__fillinf__COVENANTA_ZHUD                                509505 non-null  float32
 5   scaler__fillnamed__fillinf__DAYS                                          509505 non-null  float32
 6   scaler__fillnamed__fillinf__DIRECT_SERVICE          


### Важное различие

`full["prediction"]` и признаки в `full` корректны для анализа, но формируются разными честными механизмами:

- `TRAIN prediction` = `oof_pred` от outer `StratifiedGroupKFold`;
- `OOS prediction` = ансамбль outer-fold моделей;
- `TRAIN features` = OOF-признаки отдельного final/report encoder;
- `OOS features` = transform того же final/report encoder, fitted на всём train.

Это сделано специально: один глобальный supervised encoder нельзя использовать до outer CV, иначе target holdout-fold может попасть в обучение.


## 20. Доверительный интервал калибровки

In [38]:
def calibration_diagnostic_group_bootstrap(
    df,
    target_col,
    pred_col,
    group_col,
    num_groups=10,
    n_boot=2000,
    random_state=42,
):
    data = df[
        [target_col, pred_col, group_col]
    ].dropna().copy()
    # --------------------------------------------------
    # 1. Те же prediction quantile bins, что в M4.2
    # --------------------------------------------------
    q = min(
        num_groups,
        data[pred_col].nunique()
    )
    data["_bin"] = pd.qcut(
        data[pred_col],
        q=q,
        labels=False,
        duplicates="drop",
    )
    data = data.dropna(subset=["_bin"])
    data["_bin"] = data["_bin"].astype(int)
    n_bins = data["_bin"].nunique()
    # --------------------------------------------------
    # 2. Обычная статистика по бинам
    # --------------------------------------------------
    grouped = (
        data.groupby("_bin")
        .agg(
            n=(target_col, "size"),
            mean_true=(target_col, "mean"),
            mean_pred=(pred_col, "mean"),
            n_groups=(group_col, "nunique"),
        )
        .reset_index()
    )
    grouped["gap"] = (
        grouped["mean_true"]
        - grouped["mean_pred"]
    )
    # --------------------------------------------------
    # 3. Тот CI, который примерно использует текущий тест
    # --------------------------------------------------
    p = grouped["mean_pred"].clip(0, 1)
    grouped["naive_se"] = np.sqrt(
        p * (1 - p) / grouped["n"]
    )
    grouped["naive_low"] = (
        grouped["mean_pred"]
        - 1.96 * grouped["naive_se"]
    ).clip(lower=0)
    grouped["naive_high"] = (
        grouped["mean_pred"]
        + 1.96 * grouped["naive_se"]
    )
    grouped["naive_pass"] = (
        (grouped["mean_true"] >= grouped["naive_low"])
        &
        (grouped["mean_true"] <= grouped["naive_high"])
    )
    # --------------------------------------------------
    # 4. Готовим group-level вклад в каждый bin
    # --------------------------------------------------
    groups = data[group_col].unique()
    group_to_idx = {
        g: i for i, g in enumerate(groups)
    }
    n_groups = len(groups)
    counts = np.zeros(
        (n_groups, n_bins),
        dtype=float
    )
    sum_y = np.zeros_like(counts)
    sum_p = np.zeros_like(counts)
    for (g, b), part in data.groupby(
        [group_col, "_bin"]
    ):
        gi = group_to_idx[g]
        counts[gi, b] = len(part)
        sum_y[gi, b] = part[target_col].sum()
        sum_p[gi, b] = part[pred_col].sum()
    # --------------------------------------------------
    # 5. Cluster bootstrap:
    #    ресемплируем целиком GROUP
    # --------------------------------------------------
    rng = np.random.default_rng(
        random_state
    )
    boot_gap = np.full(
        (n_boot, n_bins),
        np.nan
    )
    for b in range(n_boot):
        # Эквивалентно выборке n_groups групп
        # с возвращением
        multiplicities = rng.multinomial(
            n_groups,
            np.full(
                n_groups,
                1 / n_groups
            )
        )
        boot_n = multiplicities @ counts
        boot_y = multiplicities @ sum_y
        boot_p = multiplicities @ sum_p
        valid = boot_n > 0
        mean_y = np.full(n_bins, np.nan)
        mean_p = np.full(n_bins, np.nan)
        mean_y[valid] = (
            boot_y[valid]
            / boot_n[valid]
        )
        mean_p[valid] = (
            boot_p[valid]
            / boot_n[valid]
        )
        boot_gap[b] = mean_y - mean_p
    # --------------------------------------------------
    # 6. Bootstrap CI непосредственно для calibration gap
    # --------------------------------------------------
    grouped["boot_gap_low"] = np.nanquantile(
        boot_gap,
        0.025,
        axis=0
    )
    grouped["boot_gap_high"] = np.nanquantile(
        boot_gap,
        0.975,
        axis=0
    )
    grouped["boot_gap_se"] = np.nanstd(
        boot_gap,
        axis=0,
        ddof=1
    )
    # Если 0 входит в CI для gap:
    # данных недостаточно, чтобы утверждать,
    # что mean_true != mean_pred
    grouped["group_bootstrap_pass"] = (
        (grouped["boot_gap_low"] <= 0)
        &
        (grouped["boot_gap_high"] >= 0)
    )
    # Насколько cluster SE больше наивного
    grouped["se_ratio"] = (
        grouped["boot_gap_se"]
        / grouped["naive_se"]
    )
    return grouped


In [46]:
calib_diag = calibration_diagnostic_group_bootstrap(
    full.loc[(full['sample'] == 'OOS')],
    target_col=TARGET_NAME,
    pred_col="prediction",
    group_col='GROUP',
    num_groups=10,
    n_boot=2000,
)
calib_diag

,_bin,n,mean_true,mean_pred,n_groups,gap,naive_se,naive_low,naive_high,naive_pass,boot_gap_low,boot_gap_high,boot_gap_se,group_bootstrap_pass,se_ratio
0,0,10145,0.001479,0.022883,35,-0.021405,0.001485,0.019973,0.025793,False,-0.022689,-0.020088,0.000666,False,0.448944
1,1,10191,0.007850,0.036434,123,-0.028584,0.001856,0.032796,0.040071,False,-0.031798,-0.024515,0.001891,False,1.018817
2,2,10090,0.016947,0.047255,204,-0.030308,0.002112,0.043115,0.051395,False,-0.038082,-0.020523,0.004458,False,2.110516
3,3,10188,0.020514,0.054064,244,-0.033550,0.002240,0.049673,0.058455,False,-0.040892,-0.025597,0.003930,False,1.754205
4,4,10092,0.036663,0.059523,246,-0.022860,0.002355,0.054906,0.064139,False,-0.032899,-0.012384,0.005387,False,2.287309
5,5,10141,0.049009,0.065541,260,-0.016532,0.002458,0.060724,0.070358,False,-0.027551,-0.005206,0.005703,False,2.320510
6,6,10141,0.069618,0.072030,270,-0.002411,0.002567,0.066998,0.077062,True,-0.016337,0.013345,0.007502,True,2.922180
7,7,10147,0.079531,0.079885,290,-0.000354,0.002691,0.074610,0.085160,True,-0.013082,0.013387,0.006593,True,2.449433
8,8,10135,0.122052,0.091977,302,0.030075,0.002871,0.086351,0.097604,False,0.013048,0.049362,0.009358,False,3.260008
9,9,10141,0.288532,0.159284,399,0.129248,0.003634,0.152162,0.166407,False,0.099173,0.161758,0.015918,False,4.380440


## 21. Двупараметрическая рекалибровка коэффициентов фичей

In [47]:
eps = 1e-8

In [48]:
y_train = np.asanyarray(
    train_data[TARGET_NAME],
    dtype=float
).ravel()

oof_p = np.asarray(oof_p, dtype=float).ravel()
test_p = np.asarray(test_p, dtype=float).ravel()

In [51]:
p_clip = np.clip(oof_p, eps, 1-eps)
logit_oof = np.log(p_clip / (1-p_clip))

X_cal = sm.add_constant(
    logit_oof,
    has_constant='add'
)

cal_model = sm.GLM(
    y_train,
    X_cal,
    family=sm.families.Binomial()
).fit()

In [52]:
a, b = cal_model.params

print('a =', a)
print('b =', b)

a = 0.20424926433844182
b = 1.1159472280129503


In [53]:
eta_oof_cal = a + b * logit_oof

oof_p_cal = 1 / (1 + np.exp(-eta_oof_cal))

In [54]:
p_test_clip = np.clip(
    test_p,
    eps,
    1 - eps
)

logit_test = np.log(
    p_test_clip / (1-p_test_clip)
)

eta_test_cal = a + b * logit_test

test_p_cal = 1 / (1 + np.exp(-eta_test_cal))

Дособираем, оригинальные вероятности пока не будем удалять

In [55]:
train_mask = full['sample'].eq('TRAIN')
oos_mask = full['sample'].eq('OOS')

full.loc[train_mask, 'prediction_calibrated'] = oof_p_cal
full.loc[oos_mask, 'prediction_calibrated'] = test_p_cal

In [ ]:
calib_diag = calibration_diagnostic_group_bootstrap(
    full.loc[(full['sample'] == 'OOS')],
    target_col=TARGET_NAME,
    pred_col="prediction_calibrated",
    group_col='GROUP',
    num_groups=10,
    n_boot=2000,
)
calib_diag

,_bin,n,mean_true,mean_pred,n_groups,gap,naive_se,naive_low,naive_high,naive_pass,boot_gap_low,boot_gap_high,boot_gap_se,group_bootstrap_pass,se_ratio
0,0,10145,0.001479,0.018275,35,-0.016797,0.001330,0.015669,0.020882,False,-0.017976,-0.015590,0.000610,False,0.458903
1,1,10191,0.007850,0.030781,123,-0.022931,0.001711,0.027428,0.034135,False,-0.026131,-0.018848,0.001888,False,1.103182
2,2,10090,0.016947,0.041183,204,-0.024236,0.001978,0.037306,0.045061,False,-0.032007,-0.014454,0.004457,False,2.253115
3,3,10188,0.020514,0.047901,244,-0.027387,0.002116,0.043754,0.052048,False,-0.034729,-0.019434,0.003930,False,1.857600
4,4,10092,0.036663,0.053366,246,-0.016703,0.002237,0.048980,0.057751,False,-0.026741,-0.006227,0.005387,False,2.407770
5,5,10141,0.049009,0.059463,260,-0.010454,0.002348,0.054860,0.064066,False,-0.021474,0.000871,0.005702,True,2.428230
6,6,10141,0.069618,0.066114,270,0.003504,0.002467,0.061278,0.070950,True,-0.010418,0.019262,0.007502,True,3.040270
7,7,10147,0.079531,0.074265,290,0.005265,0.002603,0.069164,0.079367,False,-0.007456,0.019009,0.006592,True,2.532394
8,8,10135,0.122052,0.087006,302,0.035046,0.002800,0.081519,0.092493,False,0.018015,0.054342,0.009357,False,3.342318
9,9,10141,0.288532,0.161781,399,0.126751,0.003657,0.154613,0.168948,False,0.097104,0.158887,0.015759,False,4.309376


In [57]:
calib_diag = calibration_diagnostic_group_bootstrap(
    full.loc[(full['sample'] == 'TRAIN')],
    target_col=TARGET_NAME,
    pred_col="prediction",
    group_col='GROUP',
    num_groups=10,
    n_boot=2000,
)
calib_diag

,_bin,n,mean_true,mean_pred,n_groups,gap,naive_se,naive_low,naive_high,naive_pass,boot_gap_low,boot_gap_high,boot_gap_se,group_bootstrap_pass,se_ratio
0,0,40568,0.001997,0.023566,144,-0.021570,0.000753,0.022090,0.025042,False,-0.022310,-0.020811,0.000381,False,0.505348
1,1,40560,0.007273,0.036373,476,-0.029100,0.000930,0.034551,0.038195,False,-0.030627,-0.027383,0.000832,False,0.895521
2,2,40564,0.018366,0.047120,687,-0.028754,0.001052,0.045058,0.049182,False,-0.032271,-0.025140,0.001796,False,1.707392
3,3,40592,0.028429,0.053916,876,-0.025487,0.001121,0.051719,0.056113,False,-0.030801,-0.020167,0.002609,False,2.327396
4,4,40568,0.033672,0.060147,1033,-0.026475,0.001180,0.057833,0.062461,False,-0.031406,-0.021493,0.002466,False,2.088895
5,5,40556,0.046430,0.066463,1089,-0.020033,0.001237,0.064039,0.068887,False,-0.026013,-0.013630,0.003147,False,2.543924
6,6,40538,0.065371,0.072951,1159,-0.007581,0.001292,0.070420,0.075483,False,-0.015838,0.001735,0.004513,True,3.494201
7,7,40562,0.086978,0.080927,1233,0.006051,0.001354,0.078273,0.083581,False,-0.002159,0.014750,0.004407,True,3.254253
8,8,40566,0.122615,0.093561,1336,0.029054,0.001446,0.090727,0.096395,False,0.020118,0.038896,0.004945,False,3.419936
9,9,40560,0.281016,0.202994,1628,0.078022,0.001997,0.199080,0.206909,False,-0.001228,0.138343,0.039995,True,20.025677


In [58]:
calib_diag = calibration_diagnostic_group_bootstrap(
    full.loc[(full['sample'] == 'TRAIN')],
    target_col=TARGET_NAME,
    pred_col="prediction_calibrated",
    group_col='GROUP',
    num_groups=10,
    n_boot=2000,
)
calib_diag

,_bin,n,mean_true,mean_pred,n_groups,gap,naive_se,naive_low,naive_high,naive_pass,boot_gap_low,boot_gap_high,boot_gap_se,group_bootstrap_pass,se_ratio
0,0,40568,0.001997,0.018892,144,-0.016895,0.000676,0.017567,0.020217,False,-0.017576,-0.016202,0.000353,False,0.522695
1,1,40560,0.007273,0.030725,476,-0.023451,0.000857,0.029045,0.032404,False,-0.024978,-0.021742,0.000832,False,0.971361
2,2,40564,0.018366,0.041052,687,-0.022686,0.000985,0.039121,0.042983,False,-0.026203,-0.019069,0.001796,False,1.823439
3,3,40592,0.028429,0.047755,876,-0.019325,0.001058,0.045680,0.049829,False,-0.024640,-0.014006,0.002609,False,2.464986
4,4,40568,0.033672,0.053995,1033,-0.020323,0.001122,0.051796,0.056195,False,-0.025255,-0.015341,0.002466,False,2.197471
5,5,40556,0.046430,0.060403,1089,-0.013974,0.001183,0.058085,0.062722,False,-0.019953,-0.007569,0.003146,False,2.659763
6,6,40538,0.065371,0.067065,1159,-0.001695,0.001242,0.064630,0.069500,True,-0.009952,0.007623,0.004513,True,3.632810
7,7,40562,0.086978,0.075353,1233,0.011625,0.001311,0.072784,0.077922,False,0.003419,0.020326,0.004406,False,3.362069
8,8,40566,0.122615,0.088690,1336,0.033925,0.001412,0.085923,0.091456,False,0.024986,0.043765,0.004946,False,3.503876
9,9,40560,0.281016,0.208200,1628,0.072816,0.002016,0.204248,0.212151,False,-0.009945,0.135057,0.041714,True,20.690926


In [ ]:
FINAL_ENCODER_PATH = "final_report_linear_features.pkl"

joblib.dump(
    {
        "reader": reader_final,
        "features_pipeline": feats_final,
        "feature_names": list(transformed_train_final.features),
        "target_name": TARGET_NAME,
        "group_name": GROUP_NAME,
        "start_date_name": START_DATE_NAME,
    },
    FINAL_ENCODER_PATH,
)

file_path = os.path.join(os.getcwd(), 'bin_subsample_ACT.csv')
full.to_csv(file_path, index=False)
print(f"Файл сохранен по пути: {file_path}")

print("Сохранён encoder:", FINAL_ENCODER_PATH)